Load imports

In [1]:
import heapq
import random
import time
import tkinter as tk
from tkinter import messagebox, simpledialog

A\* logic and implementation

In [ ]:

goal = (1, 2, 3, 4, 5, 6, 7, 8, 0) 

goal_pos = {goal[i]: (i // 3, i % 3) for i in range(9)}


def manhattan(state): #we are using manhattan distance
    dist = 0
    for i, tile in enumerate(state):
        if tile != 0:
            gr, gc = goal_pos[tile]
            cr, cc = i // 3, i % 3
            dist += abs(cr - gr) + abs(cc - gc)
    return dist


def get_neighbors(state):
    neighbors = []
    blank = state.index(0)
    r, c = blank // 3, blank % 3

    moves = [("Up", -1,0),
             ("Down",1, 0),
             ("Left",0, -1),
             ("Right",0, 1)]

    for label, dr, dc in moves:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            swap = nr * 3 + nc
            lst = list(state)
            lst[blank], lst[swap] = lst[swap], lst[blank]
            neighbors.append((label, tuple(lst)))
    return neighbors


def astar(start):
    t0 = time.perf_counter()

    h0 = manhattan(start)
    open_heap = [(h0, 0, start, [("Start", start)])]
    
    closed = {}
    nodes_expanded = 0

    while open_heap:
        f, g, state, path = heapq.heappop(open_heap)

        if state in closed and closed[state] <= g: #skip if cheaper state found already
            continue
        closed[state] = g
        nodes_expanded += 1
        
        if state == goal:
            return path, nodes_expanded, time.perf_counter() - t0

        for label, neighbor in get_neighbors(state):
            ng = g + 1
            if neighbor not in closed or closed[neighbor] > ng:
                nh = manhattan(neighbor)
                heapq.heappush(open_heap,
                               (ng + nh, ng, neighbor, path + [(label, neighbor)]))

    return None, nodes_expanded, time.perf_counter() - t0

def is_solvable(state): 
    tiles = [t for t in state if t != 0]
    inv = sum(1 for i in range(len(tiles))
                for j in range(i + 1, len(tiles))
                if tiles[i] > tiles[j])
    return inv% 2 == 0

def random_puzzle():
    state = list(goal)
    while True:
        random.shuffle(state)
        t = tuple(state)
        if is_solvable(t) and t != goal:
            return t

print("A* core logic done")

A* core logic done


GUI using tkinter

In [3]:
BG  = "#1e1e2e"
TILE_BG = "#313244"
TILE_FG  = "#cdd6f4"
BLANK_BG = "#1e1e2e"
ACTIVE_BG= "#89b4fa"   
ACTIVE_FG="#1e1e2e"
goal_BG= "#a6e3a1"   
goal_FG = "#1e1e2e"
BTN_BG = "#45475a"
BTN_FG = "#cdd6f4"
BTN_ACT="#585b70"
INFO_FG  ="#bac2de"

TILE_SIZE  = 110
ANIM_DELAY = 500  

class PuzzleGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("8-Puzzle — A* Solver")
        self.root.configure(bg=BG)
        self.root.resizable(False, False)

        self.state = None  
        self.solution = []    
        self.step_index = 0
        self.animating = False
        self.after_id = None

        self._build_ui()
        self._new_random()

    def _build_ui(self):
        #title
        tk.Label(self.root, text="8-Puzzle  ·  A* Solver",
                 font=("Helvetica", 18, "bold"),
                 bg=BG, fg="#cba6f7").pack(pady=(18, 4))

        #info bar
        self.info_var = tk.StringVar(value="Press Solve to start.")
        tk.Label(self.root, textvariable=self.info_var,
                 font=("Helvetica", 11), bg=BG, fg=INFO_FG,
                 wraplength=370).pack(pady=(0, 8))

        canvas_size = TILE_SIZE * 3 + 16
        self.canvas = tk.Canvas(self.root,
                                width=canvas_size, height=canvas_size,
                                bg=BG, highlightthickness=0)
        self.canvas.pack(padx=20)

        self.step_var = tk.StringVar(value="")
        tk.Label(self.root, textvariable=self.step_var,
                 font=("Helvetica", 10), bg=BG, fg=INFO_FG).pack(pady=4)

        btn_frame = tk.Frame(self.root, bg=BG)
        btn_frame.pack(pady=(4, 6))

        btns = [
            ("Random", self._new_random),
            ("Manual", self._manual_input),
            ("▶  Solve",self._solve),
            ("⏭  Step", self._step_forward),
            ("Reset",self._reset_to_start),
        ]
        for text, cmd in btns:
            tk.Button(btn_frame, text=text, command=cmd,
                      bg=BTN_BG, fg=BTN_FG,
                      activebackground=BTN_ACT, activeforeground=BTN_FG,
                      relief="flat", padx=10, pady=6,
                      font=("Helvetica", 10)).pack(side="left", padx=4)

        # Speed slider
        spd_frame = tk.Frame(self.root, bg=BG)
        spd_frame.pack(pady=(2, 14))
        tk.Label(spd_frame, text="Animation speed:",
                 bg=BG, fg=INFO_FG,
                 font=("Helvetica", 9)).pack(side="left")
        self.speed_var = tk.IntVar(value=ANIM_DELAY)
        tk.Scale(spd_frame, from_=100, to=1500,
                 orient="horizontal", variable=self.speed_var,
                 bg=BG, fg=INFO_FG, troughcolor=TILE_BG,
                 highlightthickness=0, length=180,
                 label="", showvalue=False).pack(side="left", padx=6)
        tk.Label(spd_frame, text="slow",
                 bg=BG, fg=INFO_FG, font=("Helvetica", 9)).pack(side="left")

    def _draw(self, state, highlight_idx=None, goal_reached=False):
        """Render the puzzle grid onto the canvas."""
        self.canvas.delete("all")
        pad = 8
        for i, tile in enumerate(state):
            r, c = i // 3, i % 3
            x0 = pad + c * TILE_SIZE
            y0 = pad + r * TILE_SIZE
            x1 = x0 + TILE_SIZE - 4
            y1 = y0 + TILE_SIZE - 4

            if tile == 0:
                bg = BLANK_BG
                fg = BLANK_BG
            elif goal_reached:
                bg = goal_BG
                fg = goal_FG
            elif i == highlight_idx:
                bg = ACTIVE_BG
                fg = ACTIVE_FG
            else:
                bg = TILE_BG
                fg = TILE_FG

            self.canvas.create_rectangle(x0, y0, x1, y1,
                                         fill=bg, outline="#585b70",
                                         width=2)
            if tile != 0:
                self.canvas.create_text((x0 + x1) // 2, (y0 + y1) // 2,
                                        text=str(tile),
                                        font=("Helvetica", 30, "bold"),
                                        fill=fg)

    def _highlight_moved_tile(self, prev_state, curr_state):
        for i in range(9):
            if prev_state[i] != curr_state[i] and curr_state[i] != 0:
                return i
        return None
    
    def _new_random(self):
        self._cancel_animation()
        self.state = random_puzzle()
        self.solution = []
        self.step_index = 0
        self._draw(self.state)
        self.info_var.set(f"Random puzzle loaded.  h = {manhattan(self.state)}")
        self.step_var.set("")

    def _manual_input(self):
        self._cancel_animation()
        prompt = ("Enter a permutation of digits 0–8, comma-separated.\n"
                  "0 represents the blank tile.\n\n"
                  "Example: 1,2,3,4,0,5,7,8,6")
        raw = simpledialog.askstring("Manual Input", prompt, parent=self.root)
        if raw is None:
            return
        try:
            nums = tuple(int(x.strip()) for x in raw.split(","))
            assert sorted(nums) == list(range(9)), "Must use digits 0-8 exactly once."
        except Exception as e:
            messagebox.showerror("Invalid Input", str(e))
            return

        if not is_solvable(nums):
            messagebox.showerror("Unsolvable",
                                 "This configuration has an odd number of inversions\n"
                                 "and cannot be solved. Please try another.")
            return

        if nums == goal:
            messagebox.showinfo("Already Solved", "That is already the goal state!")
            return

        self.state  = nums
        self.solution = []
        self.step_index = 0
        self._draw(self.state)
        self.info_var.set(f"Manual puzzle loaded.  h = {manhattan(self.state)}")
        self.step_var.set("")

    def _solve(self):
        if self.state is None:
            return
        self._cancel_animation()
        self.info_var.set("Running A* …")
        self.root.update()

        path, nodes, elapsed = astar(self.state)

        if path is None:
            messagebox.showerror("No Solution", "No solution found.")
            return

        self.solution = path          
        self.step_index = 0
        moves = len(path) - 1
        self.info_var.set(
            f"Solved in {moves} moves  |  {nodes} nodes expanded  |  {elapsed*1000:.1f} ms"
        )
        self._animate()

    def _animate(self):
        if self.step_index >= len(self.solution):
            self.animating = False
            return

        self.animating = True
        label, state = self.solution[self.step_index]
        total = len(self.solution) - 1
        goal_reached=(state == goal)

        if self.step_index > 0:
            prev_state = self.solution[self.step_index - 1][1]
            highlight = self._highlight_moved_tile(prev_state, state)
        else:
            highlight = None

        self._draw(state, highlight_idx=highlight, goal_reached=goal_reached)
        step_num = self.step_index 
        self.step_var.set(
            f"Step {step_num} / {total} | Move: {label} | h(n) = {manhattan(state)}"
        )

        self.step_index += 1
        if self.step_index < len(self.solution):
            self.after_id = self.root.after(self.speed_var.get(), self._animate)
        else:
            self.animating = False
            self.step_var.set(f"goal reached in {total} moves!")

    def _step_forward(self):
        if not self.solution:
            self._solve()
            return
        self._cancel_animation()  
        if self.step_index >= len(self.solution):
            return
        self._animate_single()

    def _animate_single(self):
        label, state = self.solution[self.step_index]
        total = len(self.solution) - 1
        goal_reached = (state == goal)

        if self.step_index > 0:
            prev_state = self.solution[self.step_index - 1][1]
            highlight = self._highlight_moved_tile(prev_state, state)
        else:
            highlight = None

        self._draw(state, highlight_idx=highlight, goal_reached=goal_reached)
        step_num = self.step_index
        self.step_var.set(
            f"Step {step_num} / {total} | Move: {label} | h(n) = {manhattan(state)}"
        )
        self.step_index += 1
        if self.step_index >= len(self.solution):
            self.step_var.set(f"goal reached in {total} moves!")

    def _reset_to_start(self):
        self._cancel_animation()
        if self.solution:
            self.step_index = 0
            init_state = self.solution[0][1]
            self._draw(init_state)
            self.step_var.set("Reset to start. Press ▶ Solve or ⏭ Step.")
        elif self.state:
            self._draw(self.state)

    def _cancel_animation(self):
        if self.after_id:
            self.root.after_cancel(self.after_id)
            self.after_id = None
        self.animating = False

print("GUI class loaded")

GUI class loaded


to run the GUI, execute this cell


In [8]:
root = tk.Tk()
app = PuzzleGUI(root)
root.mainloop()

Testing without GUI, on cli

In [7]:
def print_state(state, label=""):
    if label:
        print(f"  [{label}]")
    for r in range(3):
        row = state[r*3 : r*3+3]
        print("  " + "  ".join(str(t) if t != 0 else "_" for t in row))
    print()

test_state = random_puzzle()
print("Initial state:")
print_state(test_state)
print(f"Manhattan distance h(start) = {manhattan(test_state)}\n")

path, nodes, elapsed = astar(test_state)

if path:
    print(f"Solution found in {len(path)-1} moves | {nodes} nodes expanded | {elapsed*1000:.2f} ms\n")
    print("Solution path:")
    for label, state in path:
        print_state(state, label)
else:
    print("No solution found.")

Initial state:
  2  3  5
  7  _  1
  4  8  6

Manhattan distance h(start) = 10

Solution found in 16 moves | 193 nodes expanded | 3.18 ms

Solution path:
  [Start]
  2  3  5
  7  _  1
  4  8  6

  [Right]
  2  3  5
  7  1  _
  4  8  6

  [Down]
  2  3  5
  7  1  6
  4  8  _

  [Left]
  2  3  5
  7  1  6
  4  _  8

  [Left]
  2  3  5
  7  1  6
  _  4  8

  [Up]
  2  3  5
  _  1  6
  7  4  8

  [Right]
  2  3  5
  1  _  6
  7  4  8

  [Down]
  2  3  5
  1  4  6
  7  _  8

  [Right]
  2  3  5
  1  4  6
  7  8  _

  [Up]
  2  3  5
  1  4  _
  7  8  6

  [Up]
  2  3  _
  1  4  5
  7  8  6

  [Left]
  2  _  3
  1  4  5
  7  8  6

  [Left]
  _  2  3
  1  4  5
  7  8  6

  [Down]
  1  2  3
  _  4  5
  7  8  6

  [Right]
  1  2  3
  4  _  5
  7  8  6

  [Right]
  1  2  3
  4  5  _
  7  8  6

  [Down]
  1  2  3
  4  5  6
  7  8  _

